# CUB B0와 Taylor 30/50/70 Teacher의 Student KD

고정 MobileNetV3-Large Student에 B0, Taylor30%, 50%, 70% Teacher를 각각 연결한다. seed42/43/44마다60epoch, 총12개 KD job과720 Student epochs를 비교한다. Student-only는 계약과 완료 근거가 검증된 기존 Native 결과를 읽기 전용으로 재사용한다.

GitHub에서 노트북만 Colab으로 열면 저장소 파일은 복사되지 않는다. 먼저 공유 브랜치를 `/content/ondeviceAI`에 clone하거나 저장소를 Colab 로컬 폴더에 준비하고 `REPO_ROOT`를 그 위치로 설정한다. Teacher와 KD는 서로 다른 전용 Python 커널에서 실행한다.

1. 첫 코드 셀에서 저장소, Drive 프로젝트 루트와 입력 경로를 설정한다.
2. setup 셀은 저장소의 동결 ZIP을 우선 사용하고 SHA-256, code ID와 source manifest를 검증한다. 저장소에 ZIP이 없으면 정확한 기존 Drive ZIP을 사용한다.
3. 계획을 확인한 뒤 필요한 실행 옵션을 명시적으로 켠다. 공유 기본값은 `START_TRAINING=False`, `AUTO_RELEASE=False`이다.

원본 이미지와 가중치는 ZIP에 포함하지 않는다. 원래 Drive source와 기존 Native baseline을 읽는 계약이 있으므로 Drive 프로젝트 루트만 바꾸는 것으로 다른 계정의 입력이 준비되지는 않는다. source ID와 해시가 다르면 실행을 중단한다.

## 1. 경로와 실행 옵션

Teacher80epoch와 Teacher Test의 완료, 이전 runtime의 종료 또는 idle 상태를 실제로 확인한 뒤 KD를 시작한다. `TERMINAL_OBSERVATION`은 실행 시점의 관측으로 채운다. 빈 dict 기본값에서는 실행 gate를 통과하지 않는다.

`DRIVE_PROJECT_ROOT` 아래 `TeacherSizeKD_NoPruning/inputs`와 원래 Native baseline/initialization 참조가 필요하다. `DATA_ROOT`는 기존 이미지 폴더를 지정하는 선택값이다. 동일 계약 중단 재개에만 `RESUME=True`를 사용한다.

실행 관측 dict에는 `observed_at_utc`(ISO8601 UTC 시각), `teacher_run_and_Test_ended`, `teacher_runtime_released_or_idle`, `total_active_sessions`, `KD_sessions`, `additional_purchases`, 관측 근거인 `evidence`를 기록한다. 앞의 두 종료 확인 값은 실제 확인 후 True, 세션 수는 실제 값, `additional_purchases`는0이어야 한다. 실행 gate는 관측 후10분 이내, 전체 활성 세션1~2개 및 KD 세션1개를 요구한다. 시각이나 확인 값을 현재 상태 확인 없이 채우지 않는다.

In [ ]:
from pathlib import Path
REPO_ROOT = Path('/content/ondeviceAI')
DRIVE_PROJECT_ROOT = Path('/content/drive/MyDrive/온디바이스AI')
START_TRAINING = False
RESUME = False
RUN_TEST_AFTER_TRAINING = True
AUTO_RELEASE = False
DATA_ROOT = None
RUNTIME_RELEASE_EVIDENCE = None
STOP_AFTER_EPOCH = None
TERMINAL_OBSERVATION = {}

## 2. 동결 KD runtime과 계획 검증

저장소의 정확한 KD ZIP을 먼저 찾고 없으면 Drive ZIP을 사용한다. source manifest와 code ID를 검증하고 Native parent를 등록한 뒤 계획을 출력한다. Teacher runtime 모듈이 로드된 커널은 사용하지 않는다.

In [ ]:
from pathlib import Path
from google.colab import drive, auth
import os, sys

REPO_ROOT = Path(REPO_ROOT).expanduser().resolve()
BASE = Path(DRIVE_PROJECT_ROOT).expanduser()
if not (REPO_ROOT / 'experiments/cub_taylor_kd/bootstrap.py').is_file():
    raise FileNotFoundError('공유 브랜치를 Colab에 clone하고 REPO_ROOT를 확인하세요.')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from experiments.cub_taylor_kd.bootstrap import load_runtime
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive', timeout_ms=900000)
FOLLOWUP_CODE = load_runtime('kd', repo_root=REPO_ROOT, drive_project_root=BASE)
from lightio.contracts import verify_delivery
assert verify_delivery(FOLLOWUP_CODE)['code_id'] == '872e1a3560df5dce08a03efdf0932763299545a61921f95418a3b2a6ef9545af'
from phaseb.native import config
cfg = config('cub200')
from lightio.kd_runner import run, plan
assert Path(run.__code__.co_filename).resolve().is_relative_to(FOLLOWUP_CODE.resolve())
print('FOLLOWUP_CODE_VERIFIED', '872e1a3560df5dce08a03efdf0932763299545a61921f95418a3b2a6ef9545af')
print(plan())

## 3. Student KD와 고정 Test 평가

Student는 MobileNetV3-Large width1.0, dropout0.2, ImageNet1K V2이다. 같은 seed의 기존 Native 초기값, train batch16/effective64, LR0.0003, weight decay0.0001, warmup5와 global cosine60을 유지한다. 손실은 `0.5 × CE + 0.5 × T² × KL`, `T=4`이다. AMP16 학습 및 Teacher autocast, FP32 Student 평가 batch16을 사용한다. Teacher 회복의 `CE + 0.5 × T² × KL`과 평가 batch8은 다른 계약이다.

Teacher는 이미 잠긴 Validation 선정본을 사용한다. Student는 전체60epoch의 Validation OA, CE, 이른 epoch 순서로 선택한다. 12개 KD job 완료와 선정본 잠금 후 Test를 평가한다. Test로 Teacher나 Student를 다시 선택하지 않는다. B-0에서 정한 Hard40과 Overall을 함께 읽는다.

In [ ]:
if START_TRAINING:
    auth.authenticate_user()
    from types import SimpleNamespace
    FOLLOWUP_REPORT=run(SimpleNamespace(start_training=True,resume=RESUME,auto_release=AUTO_RELEASE,
        test_after_training=RUN_TEST_AFTER_TRAINING,terminal_observation=TERMINAL_OBSERVATION,
        data_root=DATA_ROOT,inputs=str(BASE/'TeacherSizeKD_NoPruning/inputs'),
        stop_after_epoch=STOP_AFTER_EPOCH,runtime_release_evidence=RUNTIME_RELEASE_EVIDENCE))
    print(FOLLOWUP_REPORT)
else:print('PREPARED: no Student training/Test/GPU allocation')

## 결과 읽기와 재개

report에서 job별 Teacher reference, seed, best epoch와 checkpoint receipt를 확인한다. history와 Test 결과를 보존하고3seed 평균과 표본 표준편차를 읽는다. 실제 completion marker와 strict restore 범위를 확인하며 패키지 준비 시점의 `prepared_only` 기록은 실행 완료 근거가 아니다.

progress/history는 매epoch, checkpoint는 local atomic 및 Drive900초와 단계/최종/계획중단 flush를 사용한다. computed와 durable을 구분하고15분 고정 손실 상한을 가정하지 않는다. 동일 계약의 중단 재개는 `RESUME=True`로 수행하고 owner/lock을 삭제하지 않는다. `AUTO_RELEASE` 기본값은 False이며 True일 때만 최종 저장 후 반환을 요청한다. 이미 관찰한 CUB Test는 새로운 holdout이 아니다.